In [1]:
# !pip install ipywidgets

In [2]:
"""
Cost + loss sensitivity analysis for quantum/QUBO pruning.

Goal:
    Produce a CSV table with one row per pruning candidate:
        candidate, L_i, C_i, accuracy_drop, f1_drop, loss_increase, params

Use this table as input for QUBO / Hamiltonian construction.

Expected model/data:
    - Model repo: canada-guesser/canadian_streetview_cities_models
    - Dataset repo: canada-guesser/Canadian-streetview-cities
    - Default model: ConvNeXT-tiny fine-tuned checkpoint from cnn_model/convnext_tiny_set_3_final.bin

Install extra dependencies:
    pip install torch torchvision timm huggingface_hub datasets scikit-learn pandas tqdm

Example run:
    python cost_loss_sensitivity.py --model convnext --max-samples 600 --candidate-limit 8 --batch-size 16

For only listing candidate block names:
    python cost_loss_sensitivity.py --model convnext --list-candidates
"""

from __future__ import annotations

import argparse
import copy
import math
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Tuple
import re

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import v2
from tqdm import tqdm

import timm
from datasets import load_dataset
from huggingface_hub import hf_hub_download


CLASS_NAMES = [
    "Calgary", "Charlottetown", "Edmonton", "Halifax", "Hamilton",
    "Kitchener-Waterloo", "Montreal", "Ottawa-Gatineau", "Quebec City", "Saskatoon",
    "St Johns", "Toronto", "Vancouver", "Victoria", "Winnipeg",
]


# -----------------------------
# Image preprocessing
# -----------------------------

def resize_and_pad(img: Image.Image, target_size=(320, 320)) -> Image.Image:
    """Same preprocessing style as the repo inference.py for the ConvNeXT model."""
    img = img.copy()
    img.thumbnail(target_size, Image.Resampling.LANCZOS)
    new_img = Image.new("RGB", target_size, (0, 0, 0))
    left = (target_size[0] - img.size[0]) // 2
    top = (target_size[1] - img.size[1]) // 2
    new_img.paste(img, (left, top))
    return new_img


def get_transform(model_name: str):
    if model_name == "convnext":
        return v2.Compose([
            v2.Lambda(lambda img: resize_and_pad(img)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
        ])

    if model_name == "swinv2":
        return transforms.Compose([
            transforms.Resize((192, 192)),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ])

    raise ValueError(f"Unknown model_name: {model_name}")


# -----------------------------
# Dataset
# -----------------------------

class StreetViewSubset(Dataset):
    """Small in-memory subset for fast sensitivity analysis."""

    def __init__(self, hf_dataset, transform):
        self.data = list(hf_dataset)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        img = row["image"]
        if not isinstance(img, Image.Image):
            img = Image.open(img)
        img = img.convert("RGB")
        x = self.transform(img)
        y = int(row["label"])
        return x, y


def build_dataloader(model_name: str, split: str, max_samples: int, batch_size: int, num_workers: int):
    """
    Loads only a subset because full dataset is large.
    For serious final numbers, increase max_samples or use the full test split.
    """
    transform = get_transform(model_name)
    ds = load_dataset(
        "canada-guesser/Canadian-streetview-cities",
        split=f"{split}[:{max_samples}]",
    )
    wrapped = StreetViewSubset(ds, transform)
    return DataLoader(
        wrapped,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )


# -----------------------------
# Model loading
# -----------------------------

def load_finetuned_model(model_name: str, device: torch.device):
    if model_name == "convnext":
        path = hf_hub_download(
            repo_id="canada-guesser/canadian_streetview_cities_models",
            filename="cnn_model/convnext_tiny_set_3_final.bin",
        )
        model = timm.create_model("convnext_tiny", pretrained=False, num_classes=15)
        checkpoint = torch.load(path, map_location=device, weights_only=False)
        # The checkpoint in the HF repo stores the state dict under model_state_dict.
        state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
        model.load_state_dict(state_dict)

    elif model_name == "swinv2":
        path = hf_hub_download(
            repo_id="canada-guesser/canadian_streetview_cities_models",
            filename="vit_model/swinv2_base_window12_192_0_finetuned_canadian_streetview.bin",
        )
        model = timm.create_model("swinv2_base_window12_192", pretrained=False, num_classes=15)
        model.load_state_dict(torch.load(path, map_location=device, weights_only=False))

    else:
        raise ValueError(f"Unknown model: {model_name}")

    model.to(device)
    model.eval()

    for name, param in model.named_parameters():
        print(f"{name:30} {param.numel():10,}")

    return model


# -----------------------------
# Evaluation
# -----------------------------

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> Dict[str, float]:
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction="sum")

    total_loss = 0.0
    total = 0
    preds: List[int] = []
    labels: List[int] = []

    for x, y in tqdm(loader, desc="evaluate", leave=False):
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        if hasattr(logits, "logits"):
            logits = logits.logits
        loss = criterion(logits, y)

        total_loss += float(loss.item())
        total += int(y.numel())
        preds.extend(torch.argmax(logits, dim=1).detach().cpu().tolist())
        labels.extend(y.detach().cpu().tolist())

    return {
        "loss": total_loss / max(total, 1),
        "accuracy": float(accuracy_score(labels, preds)),
        "macro_f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "n_samples": total,
    }


# -----------------------------
# Candidate selection
# -----------------------------

@dataclass
class Candidate:
    name: str
    module: nn.Module
    n_params: int


def count_trainable_params(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


def find_structured_candidates(model: nn.Module, candidate_limit: int) -> List[Candidate]:
    """
    Finds only full structured blocks, not internal layers.

    For timm ConvNeXT:
        stages.0.blocks.0
        stages.1.blocks.2
        stages.2.blocks.5

    For timm SwinV2:
        layers.0.blocks.0
        layers.1.blocks.1

    We avoid internal modules like:
        stages.2.blocks.0.mlp.fc1
        stages.2.blocks.0.mlp.fc2
        stages.2.blocks.0.conv_dw
    because bypassing those breaks tensor shapes.
    """

    block_name_patterns = [
        r"^stages\.\d+\.blocks\.\d+$",  # ConvNeXT full blocks
        r"^layers\.\d+\.blocks\.\d+$",  # Swin/SwinV2 full blocks
    ]

    raw: List[Candidate] = []

    for name, module in model.named_modules():
        n_params = count_trainable_params(module)
        if n_params == 0:
            continue

        is_full_block = any(
            re.match(pattern, name) is not None
            for pattern in block_name_patterns
        )

        if is_full_block:
            raw.append(Candidate(name=name, module=module, n_params=n_params))

    if not raw:
        print("\nNo strict block candidates found.")
        print("Available module names containing 'blocks':")
        for name, module in model.named_modules():
            if "blocks" in name:
                print(name, type(module).__name__)
        raise RuntimeError("No full block candidates found. Check model.named_modules().")

    # If too many candidates, select evenly across depth.
    if len(raw) > candidate_limit:
        indices = np.linspace(0, len(raw) - 1, candidate_limit).round().astype(int)
        raw = [raw[i] for i in indices]

    return raw

# -----------------------------
# Bypass/mask wrapper
# -----------------------------

class CandidateWrapper(nn.Module):
    """
    Wraps a model block and allows temporary bypass.

    bypass=False: normal module behavior.
    bypass=True : returns input x if shape-compatible.
                  If input/output shapes are not compatible, it falls back to zeroing output.

    For ConvNeXT/Swin blocks inside a stage, input and output shapes usually match,
    so bypass approximates removing the transformation but preserving the information path.
    """

    def __init__(self, module: nn.Module):
        super().__init__()
        self.module = module
        self.bypass = False

    def forward(self, x, *args, **kwargs):
        if not self.bypass:
            return self.module(x, *args, **kwargs)

        # Preferred pruning simulation: bypass block.
        # This is safe when input and output shapes are the same.
        return x


def _get_parent_and_child(model: nn.Module, module_name: str):
    parts = module_name.split(".")
    parent = model
    for p in parts[:-1]:
        parent = parent[int(p)] if p.isdigit() else getattr(parent, p)
    child_key = parts[-1]
    return parent, child_key


def replace_module(model: nn.Module, module_name: str, new_module: nn.Module):
    parent, child_key = _get_parent_and_child(model, module_name)
    if child_key.isdigit():
        parent[int(child_key)] = new_module
    else:
        setattr(parent, child_key, new_module)


def wrap_candidates(model: nn.Module, candidates: List[Candidate]) -> Dict[str, CandidateWrapper]:
    wrappers: Dict[str, CandidateWrapper] = {}
    for cand in candidates:
        wrapper = CandidateWrapper(cand.module)
        replace_module(model, cand.name, wrapper)
        wrappers[cand.name] = wrapper
    return wrappers


# -----------------------------
# Sensitivity analysis
# -----------------------------

def run_sensitivity(
    model: nn.Module,
    loader: DataLoader,
    candidates: List[Candidate],
    device: torch.device,
    output_csv: str,
):
    print("\nWrapping pruning candidates...")
    wrappers = wrap_candidates(model, candidates)

    print("\nEvaluating baseline model...")
    baseline = evaluate(model, loader, device)
    print(f"Baseline: loss={baseline['loss']:.6f}, acc={baseline['accuracy']:.4f}, f1={baseline['macro_f1']:.4f}, n={baseline['n_samples']}")

    rows: List[Dict[str, Any]] = []

    for cand in candidates:
        print(f"\nTesting candidate: {cand.name}  params={cand.n_params:,}")
        wrapper = wrappers[cand.name]

        # Temporarily bypass/prune this candidate only.
        wrapper.bypass = True
        metrics = evaluate(model, loader, device)
        wrapper.bypass = False

        loss_increase = metrics["loss"] - baseline["loss"]
        accuracy_drop = baseline["accuracy"] - metrics["accuracy"]
        f1_drop = baseline["macro_f1"] - metrics["macro_f1"]

        row = {
            "candidate": cand.name,
            "params": cand.n_params,
            "baseline_loss": baseline["loss"],
            "pruned_loss": metrics["loss"],
            "loss_increase_raw": loss_increase,
            "baseline_accuracy": baseline["accuracy"],
            "pruned_accuracy": metrics["accuracy"],
            "accuracy_drop_raw": accuracy_drop,
            "baseline_macro_f1": baseline["macro_f1"],
            "pruned_macro_f1": metrics["macro_f1"],
            "f1_drop_raw": f1_drop,
        }
        rows.append(row)

        print(
            f"  pruned: loss={metrics['loss']:.6f}, acc={metrics['accuracy']:.4f}, f1={metrics['macro_f1']:.4f}\n"
            f"  drops : Δloss={loss_increase:.6f}, Δacc={accuracy_drop:.4f}, Δf1={f1_drop:.4f}"
        )

    df = pd.DataFrame(rows)

    # Use validation-loss increase as L_i because it is smoother than accuracy.
    # Clamp to zero because a negative increase is usually validation noise.
    df["L_i_raw"] = df["loss_increase_raw"].clip(lower=0.0)
    df["C_i_raw"] = df["params"].astype(float)

    max_L = float(df["L_i_raw"].max())
    max_C = float(df["C_i_raw"].max())

    df["L_i"] = df["L_i_raw"] / max_L if max_L > 0 else 0.0
    df["C_i"] = df["C_i_raw"] / max_C if max_C > 0 else 0.0

    # Useful ranking: low L_i and high C_i = good pruning candidate.
    # Add epsilon to avoid division by zero.
    df["pruning_attractiveness"] = df["C_i"] / (df["L_i"] + 1e-6)
    df = df.sort_values("pruning_attractiveness", ascending=False)

    df.to_csv(output_csv, index=False)
    print(f"\nSaved cost/loss table to: {output_csv}")
    print("\nColumns for QUBO:")
    print(df[["candidate", "L_i", "C_i", "loss_increase_raw", "accuracy_drop_raw", "params", "pruning_attractiveness"]].to_string(index=False))
    return df


# -----------------------------
# Main
# -----------------------------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", choices=["convnext", "swinv2"], default="convnext")
    parser.add_argument("--split", default="test")
    parser.add_argument("--max-samples", type=int, default=600)
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--num-workers", type=int, default=0)
    parser.add_argument("--candidate-limit", type=int, default=10)
    parser.add_argument("--output-csv", default="cost_loss_table.csv")
    parser.add_argument("--list-candidates", action="store_true")

    # This fixes the Jupyter --f=kernel.json problem
    args, unknown = parser.parse_known_args()

    if unknown:
        print("Ignored unknown Jupyter arguments:", unknown)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    print(f"Loading {args.model} model...")
    model = load_finetuned_model(args.model, device)

    candidates = find_structured_candidates(model, args.candidate_limit)
    if not candidates:
        raise RuntimeError("No candidate blocks found. Print model.named_modules() and select candidates manually.")

    print("\nSelected candidate blocks:")
    for i, cand in enumerate(candidates, start=1):
        print(f"C{i}: {cand.name:<55s} params={cand.n_params:,} type={type(cand.module).__name__}")

    if args.list_candidates:
        return

    print("\nLoading validation/test subset...")
    loader = build_dataloader(
        model_name=args.model,
        split=args.split,
        max_samples=args.max_samples,
        batch_size=args.batch_size,
        num_workers=args.num_workers,
    )

    run_sensitivity(
        model=model,
        loader=loader,
        candidates=candidates,
        device=device,
        output_csv=args.output_csv,
    )


if __name__ == "__main__":
    main()


Ignored unknown Jupyter arguments: ['--f=c:\\Users\\Setare\\AppData\\Roaming\\jupyter\\runtime\\kernel-v39bef7c9dd8566219f66d985b5cf37647f1ed050d.json']
Device: cpu
Loading convnext model...


stem.0.weight                       4,608
stem.0.bias                            96
stem.1.weight                          96
stem.1.bias                            96
stages.0.blocks.0.gamma                96
stages.0.blocks.0.conv_dw.weight      4,704
stages.0.blocks.0.conv_dw.bias         96
stages.0.blocks.0.norm.weight          96
stages.0.blocks.0.norm.bias            96
stages.0.blocks.0.mlp.fc1.weight     36,864
stages.0.blocks.0.mlp.fc1.bias        384
stages.0.blocks.0.mlp.fc2.weight     36,864
stages.0.blocks.0.mlp.fc2.bias         96
stages.0.blocks.1.gamma                96
stages.0.blocks.1.conv_dw.weight      4,704
stages.0.blocks.1.conv_dw.bias         96
stages.0.blocks.1.norm.weight          96
stages.0.blocks.1.norm.bias            96
stages.0.blocks.1.mlp.fc1.weight     36,864
stages.0.blocks.1.mlp.fc1.bias        384
stages.0.blocks.1.mlp.fc2.weight     36,864
stages.0.blocks.1.mlp.fc2.bias         96
stages.0.blocks.2.gamma                96
stages.0.blocks.2.conv

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]


Wrapping pruning candidates...

Evaluating baseline model...


...

  pruned: loss=0.120547, acc=0.9883, f1=0.9875
  drops : Δloss=0.031910, Δacc=0.0033, Δf1=0.0036

Testing candidate: stages.2.blocks.5  params=1,201,920


  pruned: loss=0.115930, acc=0.9900, f1=0.9894
  drops : Δloss=0.027292, Δacc=0.0017, Δf1=0.0017

Testing candidate: stages.2.blocks.7  params=1,201,920


  pruned: loss=0.132731, acc=0.9850, f1=0.9836
  drops : Δloss=0.044094, Δacc=0.0067, Δf1=0.0075

Testing candidate: stages.3.blocks.0  params=4,763,136


  pruned: loss=0.378676, acc=0.9433, f1=0.9395
  drops : Δloss=0.290038, Δacc=0.0483, Δf1=0.0516

Testing candidate: stages.3.blocks.2  params=4,763,136


  pruned: loss=0.164050, acc=0.9717, f1=0.9699
  drops : Δloss=0.075412, Δacc=0.0200, Δf1=0.0212

Saved cost/loss table to: cost_loss_table.csv

Columns for QUBO:
        candidate      L_i      C_i  loss_increase_raw  accuracy_drop_raw  params  pruning_attractiveness
stages.2.blocks.2 0.004447 0.252338           0.012119           0.000000 1201920               56.727203
stages.3.blocks.2 0.027674 1.000000           0.075412           0.020000 4763136               36.134254
stages.2.blocks.0 0.008847 0.252338           0.024109           0.005000 1201920               28.518897
stages.2.blocks.5 0.010015 0.252338           0.027292           0.001667 1201920               25.192888
stages.2.blocks.3 0.011710 0.252338           0.031910           0.003333 1201920               21.547627
stages.2.blocks.7 0.016181 0.252338           0.044094           0.006667 1201920               15.593862
stages.3.blocks.0 0.106433 1.000000           0.290038           0.048333 4763136              